# BUỔI 04 — CỤM HADOOP, HDFS CLI VÀ THUẬT TOÁN MAPREDUCE

**Notebook thực hành dành cho học viên**

|  |  |
|:---|:---|
| **Bộ dữ liệu** | `wordcount_corpus.txt` (văn bản) · `transactions.csv` (200.000 giao dịch) |
| **Thời lượng** | 60 phút (Lý thuyết 25' + Thực hành 35') |
| **Môi trường** | Hadoop 3.2.1 trên Docker — HDFS + **YARN** |

---

## Buổi này nối tiếp Buổi 03 như thế nào

```text
BUỔI 03                                BUỔI 04
Một máy chủ PostgreSQL           ──►   Cụm nhiều máy (HDFS)
Bảng nằm trên một ổ đĩa          ──►   Tệp cắt thành khối, rải khắp cụm
GROUP BY type                    ──►   Map → Shuffle → Reduce
Bộ tối ưu truy vấn tự lo         ──►   Bạn tự viết mapper và reducer
```

> **Mục tiêu lớn nhất:** cùng một phép tính *"doanh thu theo ngành hàng"* sẽ được làm **bốn lần** bằng bốn cách hoàn toàn khác nhau — Python thuần, ống dẫn Unix, job Hadoop trên YARN, và Pandas. **Cả bốn phải ra đúng cùng một bảng số.**

---

## Cách làm việc với notebook này

Mỗi nhiệm vụ có **4 thành phần**:

1. **Nhiệm vụ & Kết quả mong đợi**
2. **PROMPT CHO AI AGENT** — dán nguyên khối vào `⌘+I` / `Ctrl+I`
3. **GỢI Ý GIẢI** — khung code dùng được ngay
4. **TỰ KIỂM TRA** — ô chấm điểm tự động

> **Trước khi bắt đầu, mở terminal ở thư mục gốc dự án và chạy:**
> ```
> docker compose up -d hadoop-namenode hadoop-datanode hadoop-resourcemanager hadoop-nodemanager hadoop-historyserver
> python3 data/make_lab_datasets.py
> ```
> Đợi 40–60 giây cho **cả năm** container đạt trạng thái `healthy`.


---
## Ô THIẾT LẬP — CHẠY Ô NÀY ĐẦU TIÊN

Ô này dò đường dẫn dữ liệu (tự sinh nếu chưa có), kiểm tra cụm HDFS + YARN, và tạo ba hàm trợ giúp:

| Hàm | Dùng để |
|:---|:---|
| `hadoop("<lệnh>")` | chạy một lệnh bất kỳ **bên trong container** NameNode |
| `hdfs("<đối số>")` | lối tắt cho `hdfs dfs <đối số>` |
| `ong_dan("<lệnh>")` | chạy một ống dẫn shell **trên máy thật** (dùng cho D5) |

**Không cần sửa gì trong ô này.**


In [ ]:
# =============================================================================
# Ô THIẾT LẬP — chạy đầu tiên, không cần chỉnh sửa
# =============================================================================
import json
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

NAMENODE   = "bigdata-hadoop-namenode"
HDFS_BASE  = "/user/bigdata/lab4"
HDFS_IN    = f"{HDFS_BASE}/input"
HDFS_OUT   = f"{HDFS_BASE}/output"
STREAM_JAR = "$HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar"


def _goc_du_an() -> Path:
    """Đi ngược cây thư mục để tìm gốc dự án (thư mục có data/make_lab_datasets.py)."""
    here = Path.cwd().resolve()
    for tm in [here, *here.parents]:
        if (tm / "data" / "make_lab_datasets.py").exists():
            return tm
    raise FileNotFoundError("Không tìm thấy gốc dự án — hãy mở notebook từ trong thư mục dự án.")


GOC        = _goc_du_an()
MR_DIR     = GOC / "lab4" / "mapreduce"
OUTPUT_DIR = GOC / "lab4" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = GOC / "data" / "raw" / "wordcount_corpus.txt"
TX_PATH     = GOC / "data" / "raw" / "transactions.csv"

# --- Đáp án chuẩn, dùng để đối chiếu ở các ô TỰ KIỂM TRA ----------------------
CHUAN_REVENUE = {
    "Gia dụng":   (32231, 58925123000),
    "Điện tử":    (36296, 57872658000),
    "Thời trang": (27778, 47277119000),
    "Thực phẩm":  (59856, 18555159000),
    "Mỹ phẩm":    (19839, 17421590000),
    "Sách":       (24000,  6693153000),
}
TONG_DOANH_THU = 206_744_802_000

if not (CORPUS_PATH.exists() and TX_PATH.exists()):
    print("Chưa có dữ liệu — đang sinh bằng data/make_lab_datasets.py ...")
    subprocess.run([sys.executable, str(GOC / "data" / "make_lab_datasets.py")], check=True)


def hadoop(lenh, hien_thi=True, im_lang_log=True):
    """Chạy một lệnh BÊN TRONG container Hadoop và trả về stdout dạng chuỗi."""
    kq = subprocess.run(["docker", "exec", NAMENODE, "bash", "-c", lenh],
                        capture_output=True, text=True)
    out, err = kq.stdout, kq.stderr
    if im_lang_log:   # lọc bớt dòng INFO/WARN của Hadoop cho dễ đọc
        err = "\n".join(d for d in err.split("\n")
                        if d.strip() and not d.startswith(("20", "WARN", "SLF4J")))
    if hien_thi:
        if out.strip():
            print(out.rstrip())
        if err.strip():
            print("[thông báo]", err.rstrip(), file=sys.stderr)
    return out


def hdfs(doi_so, **kw):
    """Lối tắt cho `hdfs dfs`.  Ví dụ: hdfs("-ls -h /user")"""
    return hadoop(f"hdfs dfs {doi_so}", **kw)


def ong_dan(lenh, hien_thi=True):
    """Chạy một ống dẫn shell TRÊN MÁY THẬT (không phải trong container)."""
    kq = subprocess.run(lenh, shell=True, capture_output=True, text=True, cwd=str(GOC))
    if hien_thi and kq.stdout.strip():
        print(kq.stdout.rstrip())
    if hien_thi and kq.returncode != 0 and kq.stderr.strip():
        print("[lỗi]", kq.stderr.rstrip()[:500], file=sys.stderr)
    return kq.stdout


# --- Kiểm tra môi trường -----------------------------------------------------
print("=" * 78)
print("MÔI TRƯỜNG THỰC HÀNH — BUỔI 04 (HADOOP HDFS · YARN · MAPREDUCE)")
print("=" * 78)
print(f"Gốc dự án        : {GOC}")
print(f"Kho văn bản      : {CORPUS_PATH.name:<24} {CORPUS_PATH.stat().st_size/1024:>10,.1f} KB")
print(f"Nhật ký giao dịch: {TX_PATH.name:<24} {TX_PATH.stat().st_size/1024**2:>10,.2f} MB")
print(f"Mapper/Reducer   : {MR_DIR}")
print(f"Thư mục kết quả  : {OUTPUT_DIR}")
print("-" * 78)

_ps = subprocess.run(["docker", "ps", "--format", "{{.Names}}\t{{.Status}}"],
                     capture_output=True, text=True).stdout
_rows = [d for d in _ps.split("\n") if "hadoop" in d.lower()]

if not _rows:
    print("CHƯA THẤY CONTAINER HADOOP NÀO ĐANG CHẠY. Mở terminal và chạy:")
    print("   docker compose up -d hadoop-namenode hadoop-datanode \\")
    print("        hadoop-resourcemanager hadoop-nodemanager hadoop-historyserver")
else:
    for d in sorted(_rows):
        print("  ", d)
    print("-" * 78)
    print("  ", hadoop("hadoop version | head -1", hien_thi=False).strip())
    print("Giao diện web  NameNode : http://localhost:9870")
    print("Giao diện web  YARN     : http://localhost:8089   <-- mở sẵn tab này")
    print("Giao diện web  History  : http://localhost:19888")
    print("=" * 78)
    print("Sẵn sàng. Chuyển sang D1.")


---
---
# VÍ DỤ MẪU — NHÌN THẤY MAPREDUCE TRÊN 12 DÒNG

Trước khi làm việc với 200.000 dòng, hãy xem thuật toán chạy trên một mẩu dữ liệu **nhỏ tới mức nhẩm tay đối chiếu được**. Ô dưới **chạy được ngay, không có TODO**.

```text
① SPLIT    6 dòng             →  2 mảnh × 3 dòng
② MAP      mỗi dòng           →  các cặp (từ, 1)
③ SHUFFLE  gom theo khóa      →  mỗi từ một danh sách
④ REDUCE   cộng dồn mỗi khóa  →  bảng tần suất
```


In [ ]:
# --- VÍ DỤ MẪU: bốn giai đoạn MapReduce, in đủ dữ liệu trung gian ---
VAN_BAN_MAU = [
    "du lieu lon la du lieu",
    "du lieu phan tan tren nhieu may",
    "may tinh doc du lieu",
    "map rut gon du lieu",
    "reduce gom du lieu",
    "du lieu la tai san",
]

# ========== ① SPLIT ==========
manh = [VAN_BAN_MAU[0:3], VAN_BAN_MAU[3:6]]
print("① SPLIT")
for i, m in enumerate(manh):
    print(f"   Mảnh {i}: {len(m)} dòng")

# ========== ② MAP ==========
def mapper_mau(cac_dong):
    """Mỗi dòng -> NHIỀU cặp (từ, 1).  Đây là 'flatMap' — nhớ tên này cho Buổi 05."""
    return [(tu, 1) for d in cac_dong for tu in d.split()]

kq_map = [mapper_mau(m) for m in manh]
print("\n② MAP  (mỗi mảnh xử lý ĐỘC LẬP — đây là chỗ Hadoop chạy song song)")
for i, kq in enumerate(kq_map):
    print(f"   Mảnh {i} phát ra {len(kq)} cặp, 5 cặp đầu: {kq[:5]}")

# ========== ③ SHUFFLE & SORT ==========
gom = defaultdict(list)
for kq in kq_map:
    for khoa, gia_tri in kq:
        gom[khoa].append(gia_tri)
gom = dict(sorted(gom.items()))
print("\n③ SHUFFLE & SORT  (gom mọi giá trị cùng khóa về một nơi — TỐN KÉM NHẤT)")
for khoa, ds in list(gom.items())[:5]:
    print(f"   {khoa:<8} -> {ds}")
print(f"   ... tổng {len(gom)} khóa")

# ========== ④ REDUCE ==========
print("\n④ REDUCE")
for tu, ds in sorted(gom.items(), key=lambda x: (-sum(x[1]), x[0]))[:5]:
    print(f"   {tu:<8}{sum(ds):>4}")

tong_cap = sum(len(k) for k in kq_map)
print(f"\n   {tong_cap} cặp  ->  {len(gom)} dòng kết quả   (đây chính là phép RÚT GỌN)")
print("   Nhẩm tay: 'du' 6 lần, 'lieu' 6 lần, 'la' 2 lần, 'may' 2 lần.")


---
---
# D1. KIỂM TRA SỨC KHỎE CỤM HDFS VÀ YARN
### 4 phút

### Nhiệm vụ

Xác nhận **cả hai tầng** của cụm đang sống: HDFS (lưu trữ) và YARN (tính toán).

### Kết quả mong đợi

```text
Live datanodes (1)          ◄── HDFS có nút lưu trữ
Total Nodes:1  ... RUNNING  ◄── YARN có nút tính toán
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Kiểm tra sức khỏe cụm Hadoop bằng các hàm hadoop() và hdfs() đã có sẵn:
1. Chạy "hdfs dfsadmin -report | head -12" để xem trạng thái HDFS
2. Chạy "yarn node -list" để xem NodeManager của YARN
3. Chạy "hdfs dfs -df -h" để xem dung lượng cụm
4. Chạy "hdfs dfs -ls /" để xem thư mục gốc HDFS
In tiêu đề rõ ràng cho từng phần.
```

### Công cụ

`hadoop()` · `hdfs dfsadmin -report` · `yarn node -list` · `hdfs dfs -df -h`


#### GỢI Ý GIẢI — D1

```python
print("=== ① HDFS: BÁO CÁO TRẠNG THÁI ===")
hadoop("hdfs dfsadmin -report | head -12")

print("\n=== ② YARN: DANH SÁCH NODEMANAGER ===")
hadoop("yarn node -list")

print("\n=== ③ DUNG LƯỢNG CỤM ===")
hdfs("-df -h")

print("\n=== ④ THƯ MỤC GỐC HDFS ===")
hdfs("-ls /")
```

**Cách đọc `dfsadmin -report`:**

| Dòng | Ý nghĩa |
|:---|:---|
| `Configured Capacity` | Tổng dung lượng cụm nhìn thấy |
| `Present Capacity` | Dung lượng thực sự dùng được |
| `DFS Used` | Dữ liệu HDFS đang chiếm |
| `Live datanodes (N)` | **Số nút lưu trữ còn sống** — quan trọng nhất |
| `Under replicated blocks` | Số khối chưa đủ bản sao (phải bằng 0) |

> **Việc bắt buộc làm trên trình duyệt:** mở **http://localhost:8089** (YARN ResourceManager). Lúc này danh sách application đang **rỗng** — hãy để nguyên tab đó. Đến D6 nó sẽ có job chạy và bạn nhìn thấy tiến độ theo thời gian thực.

> **Vì sao mỗi dịch vụ có một web UI riêng?** Vì trong cụm thật chúng là những **tiến trình độc lập trên những máy khác nhau**. Trong Docker chúng chỉ *tình cờ* cùng nằm trên một máy.


In [ ]:
# --- D1: Kiểm tra sức khỏe cụm ---
# TODO ①: In báo cáo HDFS  -> hadoop("hdfs dfsadmin -report | head -12")
# TODO ②: In danh sách NodeManager của YARN
# TODO ③: In dung lượng cụm bằng -df -h
# TODO ④: Liệt kê thư mục gốc HDFS


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D1 (không cần sửa)
# =============================================================================
def _kiem_tra_d1():
    import re
    bc = hadoop("hdfs dfsadmin -report", hien_thi=False)
    nodes = hadoop("yarn node -list", hien_thi=False)

    m_live = re.search(r"Live datanodes \((\d+)\)", bc)
    so_dn  = int(m_live.group(1)) if m_live else 0
    m_cap  = re.search(r"Present Capacity:\s+(\d+)", bc)
    gb     = int(m_cap.group(1)) / 1024**3 if m_cap else 0
    so_nm  = nodes.count("RUNNING")

    print(f"HDFS  — Live datanodes   : {so_dn}      (kỳ vọng >= 1)")
    print(f"HDFS  — Dung lượng dùng được : {gb:.2f} GB")
    print(f"YARN  — NodeManager RUNNING  : {so_nm}      (kỳ vọng >= 1)")
    print("-" * 62)
    if so_dn >= 1 and so_nm >= 1 and gb > 0:
        print("[DAT] Cả hai tầng của cụm đều khỏe: HDFS lưu được, YARN chạy được job.")
    elif so_dn >= 1 and so_nm == 0:
        print("[CHUA DAT] HDFS sống nhưng YARN chưa có NodeManager.")
        print("           Job sẽ kẹt vĩnh viễn ở trạng thái ACCEPTED. Hãy chạy:")
        print("           docker compose up -d hadoop-resourcemanager hadoop-nodemanager")
    else:
        print("[CHUA DAT] Cụm chưa sẵn sàng. Kiểm tra: docker compose ps")
        print("           Nếu vừa khởi động, NameNode có thể còn ở safe mode — đợi 30-60 giây.")

_kiem_tra_d1()


---
# D2. ĐƯA DỮ LIỆU LÊN HDFS BẰNG CLI
### 5 phút

### Quy trình BẮT BUỘC hai bước — điểm hay nhầm nhất

```text
Máy thật của bạn          Container Hadoop           HDFS
      │                          │                     │
      │  ① docker cp             │                     │
      ├─────────────────────────►│                     │
      │                          │  ② hdfs dfs -put    │
      │                          ├────────────────────►│
```

Phải qua **hai bước** vì lệnh `hdfs` chỉ tồn tại **bên trong** container.

### Kết quả mong đợi

```text
-rw-r--r--   1 root supergroup     18.5 M  /user/bigdata/lab4/input/transactions.csv
-rw-r--r--   1 root supergroup      6.8 K  /user/bigdata/lab4/input/wordcount_corpus.txt

200001   ◄── 200.000 giao dịch + 1 dòng tiêu đề
102      ◄── số dòng kho văn bản
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Đưa hai tệp dữ liệu lên HDFS, dùng các biến có sẵn CORPUS_PATH, TX_PATH,
NAMENODE, HDFS_IN và các hàm hadoop(), hdfs():
1. Dùng subprocess.run chạy "docker cp <đường dẫn> <NAMENODE>:/tmp/" cho CẢ HAI tệp
2. Tạo thư mục HDFS_IN bằng "hdfs dfs -mkdir -p"
3. Đẩy wordcount_corpus.txt lên bằng "hdfs dfs -put -f"
4. Đẩy transactions.csv lên với kích thước khối 8 MB:
   "hdfs dfs -D dfs.blocksize=8388608 -put -f /tmp/transactions.csv <HDFS_IN>/"
5. Kiểm chứng: "-ls -h", "-du -h", và đếm dòng của cả hai tệp bằng "-cat ... | wc -l"
In tiêu đề rõ ràng cho từng bước.
```

### Công cụ

`subprocess.run()` · `docker cp` · `-mkdir -p` · `-put -f` · `-D dfs.blocksize` · `-ls -h` · `-du -h`


#### GỢI Ý GIẢI — D2

```python
# ① Chép cả hai tệp từ máy thật vào container
for p in (CORPUS_PATH, TX_PATH):
    subprocess.run(["docker", "cp", str(p), f"{NAMENODE}:/tmp/"], check=True)
    print("① đã chép vào container:", p.name)

# ② Tạo thư mục trên HDFS
hdfs(f"-mkdir -p {HDFS_IN}")

# ③ Đẩy lên HDFS.  transactions.csv dùng khối 8 MB để D3 nhìn thấy chia khối
hdfs(f"-put -f /tmp/wordcount_corpus.txt {HDFS_IN}/")
hdfs(f"-D dfs.blocksize=8388608 -put -f /tmp/transactions.csv {HDFS_IN}/")

# ④ Kiểm chứng
print("\n=== LIỆT KÊ ===");   hdfs(f"-ls -h {HDFS_IN}/")
print("\n=== DUNG LƯỢNG ==="); hdfs(f"-du -h {HDFS_IN}/")
print("\n=== ĐẾM DÒNG ===")
for ten, ky_vong in [("transactions.csv", 200001), ("wordcount_corpus.txt", 102)]:
    n = hadoop(f"hdfs dfs -cat {HDFS_IN}/{ten} 2>/dev/null | wc -l", hien_thi=False).strip()
    print(f"   {ten:<24}{n:>8}   (kỳ vọng {ky_vong})")
```

**Đọc một dòng `-ls -h`:**

```text
-rw-r--r--   1   root  supergroup   18.5 M   2026-08-25 15:00   /user/.../transactions.csv
    │        │    │        │           │
    │        │    │        │           └─ dung lượng THẬT (khối cuối KHÔNG bị lấp đầy)
    │        │    │        └───────────── nhóm sở hữu
    │        │    └────────────────────── người sở hữu
    │        └─────────────────────────── SỐ BẢN SAO  ◄── để ý con số này
    └──────────────────────────────────── quyền truy cập, giống hệt Linux
```

> **Ba lỗi hay gặp:**
> 1. **Nhầm chiều `-put`/`-get`.** Mẹo nhớ: **put** = *đặt vào* HDFS, **get** = *lấy ra*.
> 2. **`put: File exists`** — chạy lần hai mà quên cờ `-f`.
> 3. **`cat: Unable to write to output stream`** — do `head`/`wc` đóng ống dẫn sớm. **Không phải lỗi.**


In [ ]:
# --- D2: Đưa dữ liệu lên HDFS ---
# TODO ①: docker cp CẢ HAI tệp (CORPUS_PATH, TX_PATH) vào container
# TODO ②: hdfs dfs -mkdir -p tạo thư mục HDFS_IN
# TODO ③: -put -f kho văn bản;  -D dfs.blocksize=8388608 -put -f nhật ký giao dịch
# TODO ④: kiểm chứng bằng -ls -h, -du -h và đếm dòng bằng -cat | wc -l


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D2 (không cần sửa)
# =============================================================================
def _kiem_tra_d2():
    ds = hadoop(f"hdfs dfs -ls {HDFS_IN}", hien_thi=False)
    dat = True
    for ten, ky_vong in [("wordcount_corpus.txt", "102"), ("transactions.csv", "200001")]:
        if ten not in ds:
            print(f"[CHUA DAT] Chưa thấy {ten} tại {HDFS_IN}")
            dat = False
            continue
        n = hadoop(f"hdfs dfs -cat {HDFS_IN}/{ten} 2>/dev/null | wc -l", hien_thi=False).strip()
        ok = (n == ky_vong)
        dat &= ok
        print(f"{'[DAT]' if ok else '[CHUA DAT]'} {ten:<24} {n:>8} dòng   (kỳ vọng {ky_vong})")
    print("-" * 62)
    if dat:
        print("Dữ liệu đã lên HDFS nguyên vẹn.")
        print("Mở http://localhost:9870 -> Utilities -> Browse the file system")
        print(f"   rồi vào {HDFS_IN} để nhìn thấy hai tệp bằng giao diện web.")
    else:
        print("Chạy lại bước ① và ③. Nhớ cờ -f để ghi đè nếu tệp đã tồn tại.")

_kiem_tra_d2()


---
# D3. KHÁM PHÁ CƠ CHẾ CHIA KHỐI
### 3 phút · **TRỌNG TÂM PHẦN HDFS**

### Nhiệm vụ

Chứng minh bằng `fsck` rằng HDFS cắt tệp thành nhiều khối, và **khối cuối không bị lấp đầy**.

```text
số khối = ceil( kích thước tệp / kích thước khối )
```

### Kết quả mong đợi

```text
dfs.blocksize   = 134,217,728 byte = 128 MB
dfs.replication = 1

wordcount_corpus.txt   6970 bytes, replication=1, 1 block(s):  OK

transactions.csv   19352039 bytes, replication=1, 3 block(s):  OK
0. blk_...1016 len=8388608 Live_repl=1
1. blk_...1017 len=8388608 Live_repl=1
2. blk_...1018 len=2574823 Live_repl=1     ◄── khối CUỐI, KHÔNG đầy
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Khám phá cơ chế chia khối của HDFS, dùng hàm hadoop() và hdfs() có sẵn:
1. Đọc "hdfs getconf -confKey dfs.blocksize" và "hdfs getconf -confKey dfs.replication",
   đổi blocksize ra MB rồi in.
2. Chạy "hdfs fsck <HDFS_IN>/wordcount_corpus.txt -files -blocks" và nhận xét số khối.
3. Chạy "hdfs fsck <HDFS_IN>/transactions.csv -files -blocks", lọc các dòng có
   "bytes," hoặc "len=" rồi in ra.
4. Dùng regex tìm mọi giá trị len= trong kết quả fsck, in bảng: khối thứ mấy,
   bao nhiêu byte, có ĐẦY hay KHÔNG, và tổng các khối có bằng kích thước tệp không.
```

### Công cụ

`hdfs getconf -confKey` · `hdfs fsck -files -blocks` · `re.findall`


#### GỢI Ý GIẢI — D3

```python
import re

# ① Cấu hình mặc định của cụm
bs = int(hadoop("hdfs getconf -confKey dfs.blocksize", hien_thi=False).strip())
rp = hadoop("hdfs getconf -confKey dfs.replication", hien_thi=False).strip()
print(f"dfs.blocksize   = {bs:,} byte = {bs/1024**2:.0f} MB")
print(f"dfs.replication = {rp}")

# ② Tệp nhỏ -> đúng 1 khối
print("\n=== FSCK KHO VĂN BẢN (7 KB) ===")
hadoop(f"hdfs fsck {HDFS_IN}/wordcount_corpus.txt -files -blocks 2>/dev/null | grep -E 'bytes,|len='")

# ③ Tệp 18,5 MB với khối 8 MB -> 3 khối
print("\n=== FSCK NHẬT KÝ GIAO DỊCH (18,5 MB, khối 8 MB) ===")
fs = hadoop(f"hdfs fsck {HDFS_IN}/transactions.csv -files -blocks", hien_thi=False)
for d in fs.split("\n"):
    if "bytes," in d or "len=" in d:
        print("   ", d.strip())

# ④ Phân tích bằng Python
lens = [int(x) for x in re.findall(r"len=(\d+)", fs)]
print("\nSố khối:", len(lens))
for i, L in enumerate(lens):
    print(f"   khối {i}: {L:>10,} byte   {'ĐẦY' if L == max(lens) else 'KHÔNG đầy  <-- phần dư'}")
print(f"Tổng các khối: {sum(lens):,} byte  (= kích thước tệp: {sum(lens) == TX_PATH.stat().st_size})")
```

**Ba điều BẮT BUỘC rút ra:**

1. **`ceil(19.352.039 / 8.388.608) = 3`** — kích thước khối là tham số **của từng tệp lúc ghi**, không phải của cả cụm. Đó là lý do `-D dfs.blocksize=...` đặt riêng được cho một lệnh `-put`.
2. **Khối cuối chỉ 2.574.823 byte.** Cộng ba khối lại đúng bằng kích thước tệp — HDFS **không đệm thêm byte rác**. Một tệp 7 KB **không** chiếm trọn 128 MB trên đĩa.
3. **Ba khối này chính là ba mảnh đầu vào của job ở D7**, nên job sẽ chạy đúng **3 map task**. Trong cụm thật, ba khối nằm trên ba máy và ba Mapper chạy song song — đó là **Data Locality**.

> **Câu hỏi mở rộng:** kích thước khối nhỏ nhất HDFS chấp nhận là bao nhiêu?
>
> **1 MB** (`dfs.namenode.fs-limits.min-block-size`). Nhỏ hơn sẽ bị từ chối, vì mỗi khối tốn ~150 byte trong **RAM của NameNode** — hàng triệu tệp nhỏ sẽ làm cạn RAM NameNode. Đó là **small files problem**.


In [ ]:
# --- D3: Khám phá cơ chế chia khối ---
# TODO ①: Đọc dfs.blocksize và dfs.replication bằng hdfs getconf -confKey
# TODO ②: fsck kho văn bản  -> quan sát: 1 khối
# TODO ③: fsck nhật ký giao dịch -> in các dòng "bytes," và "len="
# TODO ④: Dùng re.findall(r"len=(\d+)", ...) để lập bảng khối, chú ý khối CUỐI


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D3 (không cần sửa)
# =============================================================================
def _kiem_tra_d3():
    import re
    fs = hadoop(f"hdfs fsck {HDFS_IN}/transactions.csv -files -blocks", hien_thi=False)
    if "transactions.csv" not in fs:
        return print("[CHUA DAT] Chưa thấy transactions.csv trên HDFS. Quay lại D2.")

    lens = [int(x) for x in re.findall(r"len=(\d+)", fs)]
    m = re.search(r"(\d+) bytes", fs)
    if not lens or not m:
        return print("[CHUA DAT] Không đọc được thông tin khối từ fsck.")

    tong_byte = int(m.group(1))
    print(f"Kích thước tệp : {tong_byte:,} byte  ({tong_byte/1024**2:.2f} MB)")
    print(f"Số khối        : {len(lens)}   (kỳ vọng 3 nếu đã dùng -D dfs.blocksize=8388608)")
    print("-" * 66)
    for i, L in enumerate(lens):
        nhan = "  <-- khối CUỐI, không bị lấp đầy" if i == len(lens) - 1 and L < max(lens) else ""
        print(f"Khối {i}: {L:>10,} byte  ({L/1024**2:.3f} MB){nhan}")
    print("-" * 66)
    print(f"Tổng các khối  : {sum(lens):,} byte   (khớp kích thước tệp: {sum(lens) == tong_byte})")

    if len(lens) >= 2 and lens[-1] < max(lens):
        print(f"\n[DAT] Bạn đã thấy HDFS chia tệp thành {len(lens)} khối,")
        print("      và khối cuối CHỈ chiếm đúng phần dữ liệu còn lại.")
        print(f"      Hãy nhớ con số {len(lens)} — ở D7 job sẽ chạy đúng {len(lens)} map task.")
    elif len(lens) == 1:
        print("\n[CHUA DAT] Mới có 1 khối. Ở D2 bạn đã dùng -D dfs.blocksize=8388608 chưa?")

_kiem_tra_d3()


---
---
# D4. MÔ PHỎNG MAPREDUCE BẰNG PYTHON THUẦN
### 5 phút · **TRỌNG TÂM PHẦN THUẬT TOÁN**

### Nhiệm vụ

Viết **bằng tay** cả bốn giai đoạn ngay trong notebook trên kho văn bản thật, **in dữ liệu trung gian sau mỗi giai đoạn**. Bước này chưa dùng Hadoop — mục đích là *nhìn thấy* dữ liệu biến đổi.

```text
① SPLIT    102 dòng                    →  3 mảnh
② MAP      mỗi dòng → NHIỀU cặp (từ,1) →  1.144 cặp
③ SHUFFLE  gom các cặp cùng khóa       →  373 khóa
④ REDUCE   cộng dồn mỗi khóa           →  373 dòng kết quả
```

### Kết quả mong đợi

| Tổng số từ | Số từ khác nhau |
|---:|---:|
| **1.144** | **373** |

| Từ | Số lần | Từ | Số lần |
|:---|---:|:---|---:|
| một | 43 | là | 20 |
| liệu | 27 | và | 16 |
| dữ | 27 | ghi | 15 |
| của | 26 | spark | 13 |
| máy | 20 | thì | 11 |

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Mô phỏng thuật toán MapReduce bằng Python thuần trên tệp CORPUS_PATH,
chia rõ 4 giai đoạn và IN KẾT QUẢ TRUNG GIAN SAU MỖI GIAI ĐOẠN:

1. SPLIT: đọc tệp thành danh sách dòng, chia thành 3 mảnh xấp xỉ bằng nhau,
   in số dòng mỗi mảnh.

2. MAP: viết hàm mapper(cac_dong) trả về danh sách cặp (tu, 1). Quy tắc chuẩn hóa
   BẮT BUỘC (phải giống hệt lab4/mapreduce/wc_mapper.py):
     - thay các dấu câu . , ; : ! ? " ( ) [ ] bằng khoảng trắng
     - CHỈ hạ chữ hoa ASCII A-Z, KHÔNG dùng str.lower() của Python
     - tách theo khoảng trắng
   Áp dụng mapper cho từng mảnh, in số cặp mỗi mảnh sinh ra.

3. SHUFFLE & SORT: gom mọi cặp theo khóa bằng collections.defaultdict(list),
   sắp xếp theo khóa, in số khóa tìm được.

4. REDUCE: cộng dồn từng khóa, lưu vào biến ket_qua_wc dạng {tu: so_lan}.
   In tổng số từ, số từ khác nhau và 10 từ xuất hiện nhiều nhất.

Đo thời gian toàn bộ bằng time.perf_counter() và lưu vào biến t_d4.
```

### Công cụ

`str.maketrans` · `collections.defaultdict` · `sorted()` · `time.perf_counter()`


#### GỢI Ý GIẢI — D4

```python
t0 = time.perf_counter()

# ========== ① SPLIT — cắt thành 3 mảnh, mô phỏng 3 khối HDFS ==========
dong = CORPUS_PATH.read_text(encoding="utf-8").splitlines()
SO_MANH = 3
kt = len(dong) // SO_MANH + 1
manh = [dong[i:i + kt] for i in range(0, len(dong), kt)]

print("① SPLIT")
for i, m in enumerate(manh):
    print(f"   Mảnh {i}: {len(m)} dòng")

# ========== ② MAP — mỗi mảnh chạy ĐỘC LẬP ==========
DAU_CAU  = str.maketrans({c: " " for c in '.,;:!?\"()[]'})
HA_ASCII = str.maketrans("ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")

def mapper(cac_dong):
    ket_qua = []
    for d in cac_dong:
        for tu in d.translate(DAU_CAU).translate(HA_ASCII).split():
            ket_qua.append((tu, 1))
    return ket_qua

ket_qua_map = [mapper(m) for m in manh]     # <-- chỗ này Hadoop chạy song song
print("\n② MAP")
for i, kq in enumerate(ket_qua_map):
    print(f"   Mapper {i} phát ra {len(kq):>5,} cặp — ví dụ: {kq[:3]}")
tong_cap = sum(len(k) for k in ket_qua_map)
print(f"   Tổng: {tong_cap:,} cặp")

# ========== ③ SHUFFLE & SORT ==========
gom = defaultdict(list)
for kq in ket_qua_map:
    for khoa, gia_tri in kq:
        gom[khoa].append(gia_tri)
gom = dict(sorted(gom.items()))
print(f"\n③ SHUFFLE & SORT: {tong_cap:,} cặp  ->  {len(gom)} khóa")
for khoa, ds in list(gom.items())[:3]:
    print(f"   '{khoa}' -> {len(ds)} giá trị")

# ========== ④ REDUCE ==========
ket_qua_wc = {khoa: sum(ds) for khoa, ds in gom.items()}
t_d4 = time.perf_counter() - t0

print("\n④ REDUCE")
print(f"   Tổng số từ      : {sum(ket_qua_wc.values()):,}")
print(f"   Số từ khác nhau : {len(ket_qua_wc):,}")
for tu, sl in sorted(ket_qua_wc.items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(f"      {tu:<12}{sl:>5}")
print(f"\n   Thời gian: {t_d4*1000:.1f} ms")
```

> **Vì sao bước này quan trọng dù chưa đụng Hadoop?**
>
> Vì Hadoop **che giấu** giai đoạn ③. Bạn không viết một dòng mã nào cho Shuffle — khung Hadoop tự làm. Nếu chưa từng tự viết, bạn sẽ không hiểu vì sao Shuffle là giai đoạn **tốn kém nhất**: nó phải gom mọi cặp cùng khóa về cùng một máy, tức là **truyền dữ liệu qua mạng**. Đây chính là chỗ Apache Spark tối ưu ở Buổi 05.

> **Đối chiếu với những thứ đã học:**
>
> | MapReduce | Pandas (B02) | SQL (B03) |
> |:---|:---|:---|
> | ② Map — phát ra `<khóa, giá trị>` | chọn cột | `SELECT cot_khoa, cot_gt` |
> | ③ Shuffle & Sort — gom theo khóa | `.groupby(...)` | `GROUP BY` |
> | ④ Reduce — tổng hợp mỗi nhóm | `.agg(["size","sum"])` | `COUNT(*), SUM(...)` |


In [ ]:
# --- D4: Mô phỏng MapReduce bằng Python thuần ---
# TODO ①: SPLIT   — đọc CORPUS_PATH thành danh sách dòng, chia 3 mảnh, in số dòng
# TODO ②: MAP     — hàm mapper() trả về các cặp (tu, 1) theo ĐÚNG quy tắc chuẩn hóa
# TODO ③: SHUFFLE — gom theo khóa bằng defaultdict(list), sắp xếp, in số khóa
# TODO ④: REDUCE  — cộng dồn, in tổng số từ / số từ khác nhau / top 10
#
# Lưu kết quả vào biến ket_qua_wc dạng {tu: so_lan}
# Lưu thời gian chạy vào biến t_d4 (giây)

ket_qua_wc = None    # <-- gán kết quả Reduce vào đây
t_d4       = None    # <-- gán thời gian chạy vào đây


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D4 (không cần sửa)
# =============================================================================
def _kiem_tra_d4():
    if not isinstance(ket_qua_wc, dict):
        return print("[CHUA DAT] Chưa gán biến ket_qua_wc. Hãy hoàn thành giai đoạn ④ REDUCE.")

    tong_tu   = sum(ket_qua_wc.values())
    khac_nhau = len(ket_qua_wc)
    CHUAN_TOP = [("một", 43), ("liệu", 27), ("dữ", 27), ("của", 26)]

    print(f"Tổng số từ      : {tong_tu:>6,}   (kỳ vọng 1.144)")
    print(f"Số từ khác nhau : {khac_nhau:>6,}   (kỳ vọng 373)")
    print("-" * 56)
    dat = (tong_tu == 1144) and (khac_nhau == 373)
    for tu, sl in CHUAN_TOP:
        thuc = ket_qua_wc.get(tu, 0)
        ok = (thuc == sl)
        dat &= ok
        print(f"  {'[DAT]' if ok else '[CHUA DAT]'} '{tu}' = {thuc}   (kỳ vọng {sl})")
    print("-" * 56)

    if dat:
        print("\nCHÍNH XÁC! Bạn vừa tự tay viết lại bốn giai đoạn của MapReduce.")
        print("Ở D5 và D6, cùng bộ số này phải xuất hiện lại — lần đầu qua ống dẫn Unix,")
        print("lần sau do một job THẬT chạy trên cụm YARN tính ra.")
    else:
        print("\nChưa khớp. Ba nguyên nhân thường gặp:")
        print("  1. Dùng str.lower() thay vì chỉ hạ chữ hoa ASCII (làm gộp Đổi/đổi)")
        print("  2. Quên thay dấu câu bằng khoảng trắng")
        print("  3. Chia mảnh làm mất dòng — kiểm tra sum(len(m) for m in manh) == 102")

_kiem_tra_d4()


---
# D5. MAPREDUCE PYTHON QUA ỐNG DẪN UNIX
### 5 phút

### Nhiệm vụ

Chuyển hai hàm vừa viết thành **hai chương trình độc lập** đúng hợp đồng Hadoop Streaming, rồi chạy chúng trên dữ liệu **đang nằm trên HDFS**.

### Hợp đồng của Hadoop Streaming

```text
1. ĐỌC  dữ liệu vào từ  stdin,   mỗi bản ghi một dòng
2. GHI  kết quả ra      stdout,  định dạng:  khóa <TAB> giá trị
3. TIN  rằng reducer nhận stdin ĐÃ SẮP XẾP theo khóa,
        và mọi dòng cùng khóa nằm LIỀN NHAU
```

Điều số 3 là điều đẹp nhất: reducer **không cần dictionary**, chỉ cần một biến đếm. Nhờ vậy nó xử lý được luồng dữ liệu **lớn hơn RAM của chính nó**.

Bốn tệp đã có sẵn trong `lab4/mapreduce/` — hãy **mở ra đọc trước khi chạy**:

| Tệp | Vai trò |
|:---|:---|
| `wc_mapper.py` / `wc_reducer.py` | WordCount |
| `tx_mapper.py` / `tx_reducer.py` | Doanh thu theo ngành hàng |

### Kết quả mong đợi

```text
Gia dụng     32231   58925123000
Mỹ phẩm      19839   17421590000
Sách         24000    6693153000
Thời trang   27778   47277119000
Thực phẩm    59856   18555159000
Điện tử      36296   57872658000
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chạy MapReduce bằng Python qua ống dẫn Unix, dùng hàm ong_dan() và các biến
NAMENODE, HDFS_IN, MR_DIR đã có sẵn:

1. In nội dung hai tệp lab4/mapreduce/wc_mapper.py và wc_reducer.py để đọc hiểu.

2. Chạy ống dẫn WordCount, đo thời gian bằng time.perf_counter():
   docker exec <NAMENODE> hdfs dfs -cat <HDFS_IN>/wordcount_corpus.txt 2>/dev/null
     | python3 <MR_DIR>/wc_mapper.py
     | LC_ALL=C sort
     | python3 <MR_DIR>/wc_reducer.py
   In số dòng kết quả và 5 từ xuất hiện nhiều nhất.

3. Chạy ống dẫn doanh thu với tx_mapper.py / tx_reducer.py trên transactions.csv,
   in nguyên kết quả 6 dòng.

4. Lưu tổng thời gian vào biến t_d5 và kết quả doanh thu vào biến
   ket_qua_revenue dạng {category: (so_giao_dich, doanh_thu)}.
```

### Công cụ

`ong_dan()` · `hdfs dfs -cat` · `LC_ALL=C sort` · `time.perf_counter()`


#### GỢI Ý GIẢI — D5

```python
# ① Đọc mã nguồn trước khi chạy — đây là phần quan trọng nhất của D5
print((MR_DIR / "wc_reducer.py").read_text(encoding="utf-8"))

# ② WordCount qua ống dẫn
t0 = time.perf_counter()
lenh_wc = (f"docker exec {NAMENODE} hdfs dfs -cat {HDFS_IN}/wordcount_corpus.txt 2>/dev/null"
           f" | python3 '{MR_DIR}/wc_mapper.py'"
           f" | LC_ALL=C sort"
           f" | python3 '{MR_DIR}/wc_reducer.py'")
kq_wc = ong_dan(lenh_wc, hien_thi=False)
dong_wc = [d for d in kq_wc.strip().split("\n") if d]
print("Số từ khác nhau:", len(dong_wc))
for d in sorted(dong_wc, key=lambda x: -int(x.split("\t")[1]))[:5]:
    print("   ", d.replace("\t", "  "))

# ③ Doanh thu qua ống dẫn
lenh_tx = (f"docker exec {NAMENODE} hdfs dfs -cat {HDFS_IN}/transactions.csv 2>/dev/null"
           f" | python3 '{MR_DIR}/tx_mapper.py'"
           f" | LC_ALL=C sort"
           f" | python3 '{MR_DIR}/tx_reducer.py'")
kq_tx = ong_dan(lenh_tx, hien_thi=False)
t_d5 = time.perf_counter() - t0

ket_qua_revenue = {}
print("\n=== DOANH THU THEO NGÀNH HÀNG ===")
for d in kq_tx.strip().split("\n"):
    p = d.split("\t")
    if len(p) == 3:
        ket_qua_revenue[p[0]] = (int(p[1]), int(p[2]))
        print(f"   {p[0]:<12}{int(p[1]):>8,}{int(p[2]):>18,}")
print(f"\nThời gian D5: {t_d5:.2f} giây")
```

> **`sort` trong ống dẫn CHÍNH LÀ Shuffle & Sort của Hadoop.** Đây là ý quan trọng nhất của D5. Ba chương trình nối bằng hai dấu `|` mô phỏng **chính xác** những gì cụm Hadoop làm, chỉ khác là chạy trên một máy. Vì thế:
>
> **Luôn TEST bằng ống dẫn Unix TRƯỚC KHI nộp job lên cụm.** Vài giây thay vì ba mươi giây, và lỗi đọc được ngay thay vì chôn trong 200 dòng log YARN.

> **Vì sao phải `LC_ALL=C sort`?** Thứ tự sắp xếp phụ thuộc **locale**. Locale tiếng Việt xếp chữ có dấu khác với Hadoop (vốn dùng thứ tự byte). `LC_ALL=C` buộc `sort` dùng **thứ tự byte thuần** — giống hệt Hadoop.

> **Vì sao job ở D6 dùng `awk` chứ không dùng chính hai tệp `.py` này?**
>
> Vì **container Hadoop của khóa học không cài Python** (image dựa trên Debian 9 đã hết hạn hỗ trợ). Trên cụm thật có Python, lệnh sẽ là:
> ```bash
> hadoop jar hadoop-streaming.jar -files wc_mapper.py,wc_reducer.py \
>   -mapper "python3 wc_mapper.py" -reducer "python3 wc_reducer.py" ...
> ```
> Hai tệp `.sh` dùng ở D6 chứa **cùng một thuật toán** viết bằng `awk`, và phải cho ra **kết quả giống hệt**.

> **Cái bẫy chuẩn hóa chữ hoa — có thật, hãy đọc kỹ.** `tolower()` của `mawk` **chỉ hạ chữ hoa ASCII**: `Dữ → dữ` nhưng `Đổi` giữ nguyên vì `Đ` không phải ASCII. Để hai bản khớp nhau, `wc_mapper.py` cũng **cố tình** chỉ hạ A–Z chứ không dùng `str.lower()`.
>
> Bài học nghề nghiệp: **hai chương trình phải dùng chung một quy tắc chuẩn hóa, nếu không kết quả sẽ lệch mà không hề báo lỗi.**


In [ ]:
# --- D5: MapReduce Python qua ống dẫn Unix ---
# TODO ①: In nội dung wc_mapper.py và wc_reducer.py để đọc hiểu hợp đồng Streaming
# TODO ②: Chạy ống dẫn WordCount (cat từ HDFS | wc_mapper.py | LC_ALL=C sort | wc_reducer.py)
# TODO ③: Chạy ống dẫn doanh thu với tx_mapper.py / tx_reducer.py
# TODO ④: Lưu kết quả và thời gian
#
# ket_qua_revenue dạng {category: (so_giao_dich, doanh_thu)}

ket_qua_revenue = None   # <-- gán kết quả doanh thu vào đây
t_d5            = None   # <-- gán thời gian chạy vào đây


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D5 (không cần sửa)
# =============================================================================
def _kiem_tra_d5():
    if not isinstance(ket_qua_revenue, dict):
        return print("[CHUA DAT] Chưa gán ket_qua_revenue. Hãy hoàn thành bước ③.")

    print(f"  {'category':<12}{'giao dịch':>11}{'doanh thu':>18}   Đối chiếu")
    print("  " + "-" * 60)
    dat = True
    for k, (n, r) in CHUAN_REVENUE.items():
        thuc = ket_qua_revenue.get(k)
        ok = (thuc == (n, r))
        dat &= ok
        if thuc is None:
            print(f"  {k:<12}{'—':>11}{'—':>18}   [CHUA DAT] thiếu khóa")
        else:
            print(f"  {k:<12}{thuc[0]:>11,}{thuc[1]:>18,}   {'[DAT]' if ok else '[CHUA DAT] lệch'}")
    print("  " + "-" * 60)
    tong = sum(v[1] for v in ket_qua_revenue.values())
    print(f"  {'TỔNG':<12}{sum(v[0] for v in ket_qua_revenue.values()):>11,}{tong:>18,}")
    print(f"  (kỳ vọng 200.000 giao dịch · {TONG_DOANH_THU:,} VND)")

    if dat and tong == TONG_DOANH_THU:
        print("\n[DAT] Ống dẫn Unix cho ra ĐÚNG kết quả. Đây là bằng chứng mapper/reducer viết đúng.")
        print("      Giờ mới đáng nộp job lên cụm — sang D6.")
    else:
        print("\n[CHUA DAT] Kiểm tra lại chỉ số cột trong tx_mapper.py:")
        print("      cot[4]=category · cot[6]=quantity · cot[7]=unit_price")

_kiem_tra_d5()


---
---
# D6. JOB MAPREDUCE THẬT TRÊN YARN — WORDCOUNT
### 5 phút · **TRỌNG TÂM BUỔI HỌC**

### Nhiệm vụ

Nộp job Hadoop Streaming lên **YARN**, theo dõi trên web UI, đọc bộ đếm.

```text
Dòng dữ liệu ──► [stdin] MAPPER [stdout] ──► "khóa \t giá trị"
                                                    │
                                    Hadoop tự Shuffle & Sort
                                                    │
Kết quả cuối ◄── [stdout] REDUCER [stdin] ◄────────┘
```

### Kết quả mong đợi

```text
Map input records=102        ◄── đọc vào 102 dòng văn bản
Map output records=1144      ◄── phát ra 1.144 cặp <từ, 1>
Reduce input records=1144    ◄── nhận ĐỦ — Shuffle không mất dữ liệu
Reduce input groups=373      ◄── gom còn 373 khóa
Reduce output records=373    ◄── ghi ra 373 dòng — đây chính là phép RÚT GỌN

Found 2 items
  _SUCCESS         (tệp rỗng, đánh dấu job hoàn tất)
  part-00000       (kết quả của Reducer số 0)
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chạy một job MapReduce THẬT bằng Hadoop Streaming trên YARN, dùng các hàm
hadoop(), hdfs() và biến MR_DIR, NAMENODE, HDFS_IN, HDFS_OUT, STREAM_JAR:

1. docker cp toàn bộ thư mục MR_DIR vào container tại /tmp/mr/, rồi chmod +x /tmp/mr/*.sh

2. TEST TRƯỚC bằng ống dẫn trong container (vài giây thay vì ~30 giây):
   hdfs dfs -cat <HDFS_IN>/wordcount_corpus.txt 2>/dev/null
     | /tmp/mr/wc_mapper.sh | sort | /tmp/mr/wc_reducer.sh | sort -k2 -nr | head -5

3. Xóa thư mục đầu ra cũ: hdfs dfs -rm -r -f -skipTrash <HDFS_OUT>/wordcount

4. Nộp job, đo thời gian bằng time.perf_counter():
   hadoop jar <STREAM_JAR> -D mapreduce.job.name=Lab4_WordCount
     -D mapreduce.job.reduces=1
     -files /tmp/mr/wc_mapper.sh,/tmp/mr/wc_reducer.sh
     -input <HDFS_IN>/wordcount_corpus.txt -output <HDFS_OUT>/wordcount
     -mapper wc_mapper.sh -reducer wc_reducer.sh 2>&1

5. Lọc log in ra các dòng chứa: Map input records, Map output records,
   Reduce input records, Reduce input groups, Reduce output records,
   Launched map tasks, completed successfully

6. Đọc kết quả: hdfs dfs -ls <HDFS_OUT>/wordcount và cat part-* sắp xếp giảm dần lấy 10 dòng

Lưu thời gian vào biến t_d6.
```

### Công cụ

`hadoop jar` · `hadoop-streaming.jar` · `-files` · `-mapper` · `-reducer` · `-D mapreduce.job.reduces`


#### GỢI Ý GIẢI — D6

```python
# ① Đưa mapper/reducer vào container
subprocess.run(["docker", "cp", f"{MR_DIR}/.", f"{NAMENODE}:/tmp/mr/"], check=True)
hadoop("chmod +x /tmp/mr/*.sh", hien_thi=False)
print("① đã chép mapper/reducer vào container")

# ② TEST TRƯỚC bằng ống dẫn — mẹo gỡ lỗi quan trọng nhất của buổi học
print("\n② TEST CỤC BỘ (sort đóng vai Shuffle & Sort)")
hadoop(f"hdfs dfs -cat {HDFS_IN}/wordcount_corpus.txt 2>/dev/null "
       f"| /tmp/mr/wc_mapper.sh | sort | /tmp/mr/wc_reducer.sh | sort -k2 -nr | head -5")

# ③ Hadoop TỪ CHỐI ghi đè — phải xóa output cũ
hdfs(f"-rm -r -f -skipTrash {HDFS_OUT}/wordcount", hien_thi=False)

# ④ Nộp job
print("\n④ ĐANG CHẠY JOB TRÊN YARN — mở http://localhost:8089 để xem tiến độ ...")
t0 = time.perf_counter()
nhat_ky = hadoop(
    f"hadoop jar {STREAM_JAR} "
    f"-D mapreduce.job.name=Lab4_WordCount "
    f"-D mapreduce.job.reduces=1 "
    f"-files /tmp/mr/wc_mapper.sh,/tmp/mr/wc_reducer.sh "
    f"-input {HDFS_IN}/wordcount_corpus.txt "
    f"-output {HDFS_OUT}/wordcount "
    f"-mapper wc_mapper.sh -reducer wc_reducer.sh 2>&1", hien_thi=False)
t_d6 = time.perf_counter() - t0
print(f"   xong sau {t_d6:.1f} giây")

# ⑤ Bộ đếm
print("\n⑤ BỘ ĐẾM CỦA JOB")
for d in nhat_ky.split("\n"):
    if any(k in d for k in ("Launched map tasks", "Launched reduce tasks",
                            "Map input records", "Map output records",
                            "Reduce input records", "Reduce input groups",
                            "Reduce output records", "completed successfully")):
        print("   ", d.strip())

# ⑥ Kết quả
print("\n⑥ KẾT QUẢ TRÊN HDFS")
hdfs(f"-ls {HDFS_OUT}/wordcount")
hadoop(f"hdfs dfs -cat {HDFS_OUT}/wordcount/part-* 2>/dev/null | sort -t'\t' -k2 -nr | head -10")
```

**Năm bộ đếm kể trọn câu chuyện một job — hãy giải thích được từng dòng:**

| Bộ đếm | Giá trị | Nói lên điều gì |
|:---|---:|:---|
| `Map input records` | 102 | Số dòng Mapper đọc vào — đúng bằng số dòng tệp |
| `Map output records` | 1.144 | **Một dòng sinh ra NHIỀU cặp** — đây là `flatMap` |
| `Reduce input records` | 1.144 | Bằng Map output → **Shuffle không mất dữ liệu** |
| `Reduce input groups` | 373 | Số khóa khác nhau sau khi gom |
| `Reduce output records` | 373 | Số dòng kết quả — **đây chính là phép RÚT GỌN** |

> **Việc bắt buộc làm trên trình duyệt TRONG LÚC job chạy:** mở **http://localhost:8089**, nhấn vào `Lab4_WordCount`:
> - **Progress** — Reduce **không bắt đầu từ 0%** mà nhảy lên ~33% ngay khi Map xong: đó là phần Shuffle đã hoàn tất.
> - **Attempt** — mỗi task là một lần thử; task chết sẽ hiện `attempt_..._1` và YARN tự chạy lại.
> - **Logs** — stdout/stderr của **chính mapper của bạn**. Đây là nơi *duy nhất* đọc được lỗi khi job thất bại.
>
> Job xong sẽ chuyển sang **http://localhost:19888** (JobHistory).

> **Lỗi hay gặp nhất: `Output directory already exists`.** Hadoop **cố tình** từ chối ghi đè để tránh mất kết quả cũ. Luôn `hdfs dfs -rm -r -f -skipTrash <output>` trước khi chạy lại.
>
> **Nếu `part-00000` rỗng:** Mapper không phát ra gì. Quay lại bước ② — gần như chắc chắn lỗi ở biểu thức `awk`.
>
> **Nếu job kẹt ở `ACCEPTED` mãi:** YARN không có NodeManager nào. Chạy `yarn node -list` để kiểm tra.


In [ ]:
# --- D6: Job MapReduce thật trên YARN — WordCount ---
# TODO ①: docker cp thư mục MR_DIR vào container tại /tmp/mr/, rồi chmod +x /tmp/mr/*.sh
# TODO ②: TEST TRƯỚC bằng ống dẫn trong container
# TODO ③: Xóa thư mục output cũ bằng -rm -r -f -skipTrash
# TODO ④: Nộp job bằng hadoop jar STREAM_JAR ..., đo thời gian
# TODO ⑤: Lọc log in ra các bộ đếm
# TODO ⑥: Đọc kết quả bằng -ls và -cat part-*

t_d6 = None    # <-- gán thời gian chạy job vào đây


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D6 (không cần sửa)
# =============================================================================
def _kiem_tra_d6():
    ds = hadoop(f"hdfs dfs -ls {HDFS_OUT}/wordcount", hien_thi=False)
    if "part-" not in ds:
        return print(f"[CHUA DAT] Chưa có kết quả tại {HDFS_OUT}/wordcount. Hãy hoàn thành bước ④.")

    print("Tệp đầu ra trên HDFS:")
    print(f"   _SUCCESS   : {'[DAT] có' if '_SUCCESS' in ds else '[CHUA DAT] thiếu (job chưa hoàn tất?)'}")
    print(f"   part-00000 : {'[DAT] có' if 'part-00000' in ds else '[CHUA DAT] thiếu'}")

    noi_dung = hadoop(f"hdfs dfs -cat {HDFS_OUT}/wordcount/part-*", hien_thi=False)
    doc = {}
    for d in noi_dung.strip().split("\n"):
        p = d.split("\t")
        if len(p) == 2:
            doc[p[0]] = int(p[1])

    print(f"\nSố từ khác nhau : {len(doc):>6,}   (kỳ vọng 373)")
    print(f"Tổng số từ      : {sum(doc.values()):>6,}   (kỳ vọng 1.144)")
    print("-" * 58)
    dat = (len(doc) == 373) and (sum(doc.values()) == 1144)
    for tu, sl in [("một", 43), ("liệu", 27), ("dữ", 27), ("của", 26)]:
        ok = doc.get(tu, 0) == sl
        dat &= ok
        print(f"   {'[DAT]' if ok else '[CHUA DAT]'} '{tu}' = {doc.get(tu, 0)}   (kỳ vọng {sl})")

    khop_d4 = isinstance(ket_qua_wc, dict) and ket_qua_wc == doc
    print("-" * 58)
    print(f"   {'[DAT]' if khop_d4 else '[CHUA DAT]'} Kết quả job YARN TRÙNG KHỚP kết quả Python ở D4")

    if dat and "_SUCCESS" in ds:
        print("""
XUẤT SẮC! Bạn vừa chạy một job MapReduce THẬT trên cụm YARN.

  Ba con đường hoàn toàn khác nhau — Python thuần trong RAM (D4),
  ống dẫn Unix với mapper.py (D5), và một job phân tán trên YARN (D6) —
  cùng hội tụ về một bộ số. MapReduce không còn là phép thuật.

  Câu hỏi cho D7: tệp transactions.csv có 3 khối. Job sẽ chạy mấy Mapper?""")
    else:
        print("\n[CHUA DAT] Test lại bằng ống dẫn trong container:")
        print(f"   hdfs dfs -cat {HDFS_IN}/wordcount_corpus.txt | /tmp/mr/wc_mapper.sh | sort | /tmp/mr/wc_reducer.sh")

_kiem_tra_d6()


---
# D7. JOB MAPREDUCE THẬT TRÊN YARN — DOANH THU THEO NGÀNH HÀNG
### 5 phút

### Nhiệm vụ

Chạy job thứ hai trên tệp 18,5 MB đã chia **3 khối**, rồi **giải thích số Mapper bằng số khối**.

### Kết quả mong đợi

```text
Launched map tasks=3          ◄── ĐÚNG BẰNG SỐ KHỐI đếm được ở D3
Launched reduce tasks=1
Map input records=200001      ◄── 200.000 giao dịch + 1 dòng tiêu đề
Map output records=200000     ◄── Mapper đã LỌC BỎ dòng tiêu đề
Reduce input groups=6
Reduce output records=6

Gia dụng     32231   58925123000
Điện tử      36296   57872658000
...
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chạy job MapReduce thứ hai trên YARN để tính doanh thu theo ngành hàng:

1. Xóa thư mục đầu ra cũ <HDFS_OUT>/revenue

2. Nộp job (mapper/reducer đã có trong /tmp/mr/ từ D6), đo thời gian:
   hadoop jar <STREAM_JAR> -D mapreduce.job.name=Lab4_RevenueByCategory
     -D mapreduce.job.reduces=1
     -files /tmp/mr/tx_mapper.sh,/tmp/mr/tx_reducer.sh
     -input <HDFS_IN>/transactions.csv -output <HDFS_OUT>/revenue
     -mapper tx_mapper.sh -reducer tx_reducer.sh 2>&1

3. Lọc log in ra: Launched map tasks, Launched reduce tasks, Map input records,
   Map output records, Reduce input records, Reduce input groups,
   Reduce output records, completed successfully

4. Đọc kết quả part-*, phân tích thành dict {category: (so_giao_dich, doanh_thu)}
   và in bảng có cột giá trị đơn hàng trung bình = doanh_thu / so_giao_dich

Lưu thời gian vào biến t_d7 và kết quả vào biến ket_qua_revenue_yarn.
```

### Công cụ

`hadoop jar` · `-files` · bộ đếm `Launched map tasks`


#### GỢI Ý GIẢI — D7

```python
hdfs(f"-rm -r -f -skipTrash {HDFS_OUT}/revenue", hien_thi=False)

print("ĐANG CHẠY JOB TRÊN YARN — mở http://localhost:8089 ...")
t0 = time.perf_counter()
nhat_ky = hadoop(
    f"hadoop jar {STREAM_JAR} "
    f"-D mapreduce.job.name=Lab4_RevenueByCategory "
    f"-D mapreduce.job.reduces=1 "
    f"-files /tmp/mr/tx_mapper.sh,/tmp/mr/tx_reducer.sh "
    f"-input {HDFS_IN}/transactions.csv "
    f"-output {HDFS_OUT}/revenue "
    f"-mapper tx_mapper.sh -reducer tx_reducer.sh 2>&1", hien_thi=False)
t_d7 = time.perf_counter() - t0
print(f"xong sau {t_d7:.1f} giây\n")

for d in nhat_ky.split("\n"):
    if any(k in d for k in ("Launched map tasks", "Launched reduce tasks",
                            "Map input records", "Map output records",
                            "Reduce input records", "Reduce input groups",
                            "Reduce output records", "completed successfully")):
        print("   ", d.strip())

noi_dung = hadoop(f"hdfs dfs -cat {HDFS_OUT}/revenue/part-*", hien_thi=False)
ket_qua_revenue_yarn = {}
for d in noi_dung.strip().split("\n"):
    p = d.split("\t")
    if len(p) == 3:
        ket_qua_revenue_yarn[p[0]] = (int(p[1]), int(p[2]))

print(f"\n{'category':<12}{'giao dịch':>11}{'doanh thu':>18}{'TB/đơn':>12}")
print("-" * 55)
for k, (n, r) in sorted(ket_qua_revenue_yarn.items(), key=lambda x: -x[1][1]):
    print(f"{k:<12}{n:>11,}{r:>18,}{r//n:>12,}")
```

**Ba điều BẮT BUỘC rút ra — phần đắt giá nhất của buổi học:**

1. **`Launched map tasks = 3` đúng bằng số khối ở D3.** Bằng chứng số học cho Data Locality: Hadoop tạo **một Mapper cho mỗi mảnh**, và mỗi Mapper chạy trên máy đang giữ mảnh đó. Muốn 30 Mapper thì chia tệp thành 30 khối — **mã nguồn không đổi một chữ**.

2. **`Map input 200001` nhưng `Map output 200000`.** Chênh đúng **một** dòng: dòng tiêu đề đã bị `$1 != "transaction_id"` loại bỏ. Hai bộ đếm này giúp **kiểm tra mapper có lọc đúng không** mà không cần mở dữ liệu.

3. **200.000 cặp kết tinh thành 6 dòng** — tỷ lệ rút gọn hơn **33.000 lần**. Toàn bộ mục đích của Reduce nằm ở con số này.

> **Bài toán này khác WordCount ở một điểm cốt lõi — hãy trả lời trước khi đọc tiếp:**
>
> WordCount: **một dòng vào → NHIỀU cặp ra** (102 → 1.144).
> Doanh thu: **một dòng vào → MỘT cặp ra** (200.001 → 200.000).
>
> Vì mapper WordCount tách dòng thành nhiều từ (`flatMap`), còn mapper doanh thu cho mỗi dòng một kết quả (`map`). **Hãy nhớ hai chữ này** — Buổi 05 sẽ gặp lại chúng dưới đúng cái tên đó trong API của Spark.

> **Phát hiện nghiệp vụ từ cột `TB/đơn`:** Thực phẩm có **nhiều giao dịch nhất** (59.856) nhưng doanh thu gần **thấp nhất**, vì mỗi đơn chỉ ~310 nghìn. Gia dụng bán ít hơn hẳn nhưng mỗi đơn ~1,83 triệu.
>
> **Tổng phản ánh quy mô, trung bình mới phản ánh bản chất.** Và về kỹ thuật: `AVG` **không cộng dồn được** như `SUM`/`COUNT` — Reducer phải giữ **cả tổng lẫn số đếm** rồi chia ở bước cuối.


In [ ]:
# --- D7: Job MapReduce thật trên YARN — Doanh thu theo ngành hàng ---
# TODO ①: Xóa thư mục output cũ HDFS_OUT/revenue
# TODO ②: Nộp job với tx_mapper.sh / tx_reducer.sh, đo thời gian
# TODO ③: Lọc log in ra bộ đếm — CHÚ Ý dòng "Launched map tasks"
# TODO ④: Đọc part-*, in bảng có thêm cột giá trị đơn hàng trung bình

ket_qua_revenue_yarn = None   # <-- {category: (so_giao_dich, doanh_thu)}
t_d7                 = None   # <-- thời gian chạy job


In [ ]:
# =============================================================================
# TỰ KIỂM TRA D7 (không cần sửa)
# =============================================================================
def _kiem_tra_d7():
    import re
    ds = hadoop(f"hdfs dfs -ls {HDFS_OUT}/revenue", hien_thi=False)
    if "part-" not in ds:
        return print(f"[CHUA DAT] Chưa có kết quả tại {HDFS_OUT}/revenue.")

    if not isinstance(ket_qua_revenue_yarn, dict):
        return print("[CHUA DAT] Chưa gán biến ket_qua_revenue_yarn.")

    print(f"  {'category':<12}{'giao dịch':>11}{'doanh thu':>18}   Đối chiếu")
    print("  " + "-" * 60)
    dat = True
    for k, (n, r) in sorted(CHUAN_REVENUE.items(), key=lambda x: -x[1][1]):
        thuc = ket_qua_revenue_yarn.get(k)
        ok = (thuc == (n, r))
        dat &= ok
        hien = f"{thuc[0]:>11,}{thuc[1]:>18,}" if thuc else f"{'—':>11}{'—':>18}"
        print(f"  {k:<12}{hien}   {'[DAT]' if ok else '[CHUA DAT]'}")
    print("  " + "-" * 60)

    # số khối của tệp đầu vào phải bằng số map task
    fs = hadoop(f"hdfs fsck {HDFS_IN}/transactions.csv -files -blocks", hien_thi=False)
    so_khoi = len(re.findall(r"len=(\d+)", fs))
    khop_d5 = isinstance(ket_qua_revenue, dict) and ket_qua_revenue == ket_qua_revenue_yarn

    print(f"  {'[DAT]' if khop_d5 else '[CHUA DAT]'} Kết quả YARN TRÙNG KHỚP ống dẫn Python ở D5")
    print(f"  Tệp đầu vào có {so_khoi} khối  ->  job phải chạy đúng {so_khoi} map task.")
    print("  Hãy tìm dòng 'Launched map tasks' trong log ở trên và đối chiếu.")

    if dat:
        print("\n[DAT] Bốn cách làm cho cùng một kết quả:")
        print("      Python thuần (D4) · ống dẫn Unix (D5) · YARN WordCount (D6) · YARN doanh thu (D7)")
    else:
        print("\n[CHUA DAT] Kiểm tra tx_mapper.sh: awk -F',' '$1 != \"transaction_id\" { print $5 \"\\t\" $7 * $8 }'")

_kiem_tra_d7()


---
# D8. GHI LẠI THỜI GIAN ĐỂ SO SÁNH Ở BUỔI 05
### 2 phút

### Nhiệm vụ

Lưu thời gian chạy của cả bốn cách vào `outputs/lab4_timings.json`. **Buổi 05 sẽ đọc chính tệp này** để so sánh Spark với MapReduce.

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Ghi lại kết quả và thời gian của buổi học:
1. Đối chiếu Pandas: đọc TX_PATH bằng pandas, tính groupby("category") ra
   số giao dịch và tổng doanh thu (quantity * unit_price), đo thời gian.
2. In bảng so sánh 5 cách làm kèm thời gian.
3. Ghi một dict vào OUTPUT_DIR/"lab4_timings.json" gồm các khóa:
   python_thuan_notebook_s, python_ong_dan_unix_s, yarn_wordcount_s,
   yarn_revenue_s, pandas_s, so_dong_giao_dich, ket_qua_doanh_thu,
   ket_qua_wordcount_top10
   dùng json.dump với ensure_ascii=False, indent=2
```


#### GỢI Ý GIẢI — D8

```python
import pandas as pd

# ① Đối chiếu bằng Pandas — cách thứ năm
t0 = time.perf_counter()
df = pd.read_csv(TX_PATH)
df["revenue"] = df["quantity"] * df["unit_price"]
kq_pandas = df.groupby("category").agg(n=("revenue", "size"), rev=("revenue", "sum"))
t_pandas = time.perf_counter() - t0
print(kq_pandas.sort_values("rev", ascending=False))
print(f"\nPandas: {t_pandas:.2f} giây")

# ② Bảng so sánh
print(f"\n{'Cách làm':<42}{'Thời gian':>12}")
print("-" * 54)
for ten, t in [("Pandas (một máy, trong RAM)",            t_pandas),
               ("Python thuần trong notebook (D4)",       t_d4),
               ("Ống dẫn Unix mapper.py|sort|reducer.py", t_d5),
               ("Job YARN — doanh thu 18,5 MB (D7)",      t_d7),
               ("Job YARN — WordCount 7 KB (D6)",         t_d6)]:
    print(f"{ten:<42}{t:>10.2f} s")

# ③ Ghi tệp cho Buổi 05
timings = {
    "python_thuan_notebook_s": round(t_d4, 4),
    "python_ong_dan_unix_s":   round(t_d5, 3),
    "yarn_wordcount_s":        round(t_d6, 2),
    "yarn_revenue_s":          round(t_d7, 2),
    "pandas_s":                round(t_pandas, 3),
    "so_dong_giao_dich":       200_000,
    "ket_qua_doanh_thu":       {k: list(v) for k, v in ket_qua_revenue_yarn.items()},
    "ket_qua_wordcount_top10": sorted(ket_qua_wc.items(), key=lambda x: (-x[1], x[0]))[:10],
}
(OUTPUT_DIR / "lab4_timings.json").write_text(
    json.dumps(timings, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nĐã ghi:", OUTPUT_DIR / "lab4_timings.json")
```

---

### CÂU HỎI QUAN TRỌNG NHẤT CỦA BUỔI HỌC

> **Job WordCount xử lý 7 KB và job doanh thu xử lý 18,5 MB — dữ liệu chênh nhau 2.600 lần — nhưng cả hai đều mất khoảng 30 giây. Vì sao thời gian gần như không phụ thuộc vào lượng dữ liệu?**

Hãy tự trả lời trước, rồi mới đọc bảng dưới.

| Tiêu chí | Pandas | Hadoop MapReduce |
|:---|:---|:---|
| 200.000 dòng | Vài giây | Hàng chục giây — thừa thãi |
| 10 TB | **Không chạy nổi**, tràn RAM | Chạy được, chỉ cần thêm máy |
| Giới hạn kích thước | RAM của **một** máy | Tổng dung lượng **cả cụm** |
| Một máy chết giữa chừng | Mất trắng, chạy lại từ đầu | YARN tự chạy lại task trên máy khác |
| Chi phí khởi động | Gần bằng 0 | **Hàng chục giây, cố định** |
| Độ phức tạp khi viết | Một dòng `groupby` | Hai chương trình + cấu hình job |

**Kết luận cần chốt:**

> Ở quy mô này, **gần như toàn bộ thời gian là chi phí cố định** — xin tài nguyên từ YARN, khởi động JVM cho ApplicationMaster, cho từng Mapper, cho Reducer. Thời gian tính toán thật gần bằng 0.
>
> **Hadoop KHÔNG nhanh hơn.** Hadoop **chạy được ở quy mô mà một máy không chạy nổi**, và **không chết khi phần cứng chết**.
>
> Chọn công cụ theo **quy mô dữ liệu**, không theo mức độ "hiện đại" của công nghệ. Dùng Hadoop cho 200.000 dòng là sai lầm nghề nghiệp — giống như thuê xe container để chở một thùng sữa.


In [ ]:
# --- D8: Ghi lại thời gian để so sánh ở Buổi 05 ---
# TODO ①: Đối chiếu bằng Pandas (đọc TX_PATH, groupby("category"), đo thời gian)
# TODO ②: In bảng so sánh 5 cách làm
# TODO ③: Ghi OUTPUT_DIR/"lab4_timings.json" — BẮT BUỘC, Buổi 05 sẽ đọc tệp này


#### Câu trả lời của bạn — D8

**Vì sao job 7 KB lại chậm hơn job 18,5 MB?**

*(Viết câu trả lời tại đây.)*

**Với dữ liệu của chuỗi bán lẻ trong bài, phải tích lũy bao lâu thì Pandas không kham nổi và cần đến Hadoop? Hãy ước lượng bằng số.**

*(Viết câu trả lời tại đây. Gợi ý: 200.000 dòng ≈ 18,5 MB cho 10 cửa hàng trong 5 tháng.)*


---
---
# TỔNG KẾT BUỔI HỌC

## Sản phẩm phải nộp

| # | Tệp | Nội dung |
|:--|:---|:---|
| 1 | `lab4/Lab4.ipynb` | Notebook đã chạy hết, không còn ô TODO trống |
| 2 | `lab4/outputs/lab4_timings.json` | Thời gian chạy — **bắt buộc**, Buổi 05 đọc tệp này |
| 3 | `lab4/outputs/report_lab04.md` | Báo cáo lệnh đã chạy, bộ đếm job, kết quả |
| 4 | Ảnh chụp `localhost:9870` | Tab **Overview** và **Utilities → Browse the file system** |
| 5 | Ảnh chụp `localhost:8089` | Danh sách application với **hai job FINISHED / SUCCEEDED** |

## Tiêu chí hoàn thành

- [ ] Năm container Hadoop `healthy`; `yarn node -list` có Node RUNNING
- [ ] Đẩy được hai tệp lên HDFS: đúng **200001** và **102** dòng
- [ ] `fsck` chứng minh `transactions.csv` có **3 khối**, khối cuối **2.574.823** byte
- [ ] Mô phỏng Python in đủ 4 giai đoạn: **1.144** từ / **373** từ khác nhau
- [ ] Ống dẫn Unix với `mapper.py`/`reducer.py` ra **đúng cùng bộ số**
- [ ] Hai job YARN có `_SUCCESS` và `part-00000`
- [ ] Doanh thu 6 ngành hàng khớp đáp án, tổng **206.744.802.000** VND
- [ ] Giải thích được vì sao `Launched map tasks = 3`
- [ ] Giải thích được vì sao job 7 KB chậm hơn job 18,5 MB
- [ ] Đã ghi `outputs/lab4_timings.json`

## Bảng chấm điểm

| Tiêu chí | Điểm |
|:---|---:|
| D1–D2 — Cụm HDFS + YARN khỏe, thao tác CLI đúng, dữ liệu lên HDFS nguyên vẹn | 15 |
| D3 — Chứng minh và giải thích cơ chế chia khối bằng `fsck` | 15 |
| D4 — Mô phỏng đủ 4 giai đoạn, có in kết quả trung gian | 15 |
| D5 — `mapper.py`/`reducer.py` đúng hợp đồng Streaming, kết quả khớp D4 | 15 |
| D6 — Job WordCount chạy thành công trên YARN, giải thích được 5 bộ đếm | 20 |
| D7 — Job doanh thu đúng, giải thích được `Launched map tasks = 3` | 15 |
| D8 — Ghi đủ timings, rút ra kết luận đúng về chi phí khởi động | 5 |
| **Tổng** | **100** |

---

## Lỗi thường gặp và cách xử lý

| Lỗi | Nguyên nhân | Cách xử lý |
|:---|:---|:---|
| `Output directory already exists` | Hadoop **cố tình** từ chối ghi đè | `hdfs dfs -rm -r -f -skipTrash <output>` |
| `put: File exists` | Tệp đã có trên HDFS | Thêm cờ `-f` |
| `cat: Unable to write to output stream` | `head`/`wc` đóng ống dẫn sớm | **Không phải lỗi** — thêm `2>/dev/null` |
| `Permission denied` khi chạy mapper | Script chưa có quyền thực thi | `chmod +x /tmp/mr/*.sh` **sau khi** `docker cp` |
| `python3: command not found` trong job | Container Hadoop không cài Python | Dùng bản `.sh` (awk) |
| Job kẹt mãi ở `ACCEPTED` | YARN không có NodeManager | `yarn node -list`; khởi động `hadoop-nodemanager` |
| `Name node is in safe mode` | NameNode vừa khởi động | Đợi 30–60 giây, hoặc `hdfs dfsadmin -safemode leave` |
| `part-00000` rỗng | Mapper không phát ra gì | Test riêng bằng ống dẫn |
| Kết quả thiếu đúng một dòng | Reducer quên in **khóa cuối** sau vòng lặp | Xem lại bộ khung reducer ở D5 |
| Python và awk lệch nhau | Hai bản chuẩn hóa văn bản khác nhau | Xem cái bẫy `tolower` ở D5 |
| `Connection refused` cổng 8089 | ResourceManager chưa chạy | `docker compose up -d hadoop-resourcemanager` |

> **Mẹo gỡ lỗi quan trọng nhất của buổi học:**
> ```
> hdfs dfs -cat <tệp> | mapper.sh | sort | reducer.sh
> ```
> Vài giây thay vì ba mươi giây. **Chưa test bằng ống dẫn thì đừng nộp job.**

---

## Buổi tiếp theo

| Buổi | Chủ đề | Liên hệ với hôm nay |
|:---|:---|:---|
| **05** | Apache Spark | Chạy lại **đúng hai bài toán này** trên **cùng dữ liệu**, đo xem nhanh hơn bao nhiêu lần. `map`/`flatMap`/`reduceByKey` của Spark chính là Map/Reduce bạn vừa viết tay — nhưng dữ liệu trung gian ở **trong RAM** thay vì ghi xuống đĩa |
| **06** | Data Wrangling | Đọc dữ liệu nguồn thẳng từ HDFS |
| **13** | Học máy | Tập đặc trưng huấn luyện lưu trên HDFS ở định dạng Parquet |

> Hãy giữ con số **~30 giây** trong đầu. Buổi 05 sẽ chạy đúng phép tính đó trong **vài giây**, và bạn sẽ hiểu chính xác vì sao.


In [ ]:
# =============================================================================
# NGHIỆM THU CUỐI BUỔI — chạy trước khi nộp bài (không cần sửa)
# =============================================================================
print("=" * 74)
print("BẢNG NGHIỆM THU BUỔI 04".center(74))
print("=" * 74)

import re as _re
_g   = globals()
_muc = []

# --- D1
_bc = hadoop("hdfs dfsadmin -report", hien_thi=False)
_nl = hadoop("yarn node -list", hien_thi=False)
_muc.append(("D1  HDFS có DataNode sống",            "Live datanodes (1)" in _bc))
_muc.append(("D1  YARN có NodeManager RUNNING",      "RUNNING" in _nl))

# --- D2
_ls = hadoop(f"hdfs dfs -ls {HDFS_IN}", hien_thi=False)
_muc.append(("D2  wordcount_corpus.txt trên HDFS",   "wordcount_corpus.txt" in _ls))
_n = hadoop(f"hdfs dfs -cat {HDFS_IN}/transactions.csv 2>/dev/null | wc -l", hien_thi=False).strip()
_muc.append((f"D2  transactions.csv đủ 200001 dòng (được {_n or '—'})", _n == "200001"))

# --- D3
_fs   = hadoop(f"hdfs fsck {HDFS_IN}/transactions.csv -files -blocks", hien_thi=False)
_lens = [int(x) for x in _re.findall(r"len=(\d+)", _fs)]
_muc.append((f"D3  Tệp chia nhiều khối (được {len(_lens)} khối)", len(_lens) >= 2))
_muc.append(("D3  Khối cuối không bị lấp đầy", bool(_lens) and _lens[-1] < max(_lens)))

# --- D4
_wc = _g.get("ket_qua_wc")
_muc.append(("D4  Mô phỏng Python: 1.144 từ / 373 từ khác nhau",
             isinstance(_wc, dict) and len(_wc) == 373 and sum(_wc.values()) == 1144))

# --- D5
_rv = _g.get("ket_qua_revenue")
_muc.append(("D5  Ống dẫn Python cho đúng doanh thu 6 ngành hàng",
             isinstance(_rv, dict) and _rv == CHUAN_REVENUE))

# --- D6
_o6 = hadoop(f"hdfs dfs -ls {HDFS_OUT}/wordcount", hien_thi=False)
_muc.append(("D6  Job WordCount có tệp _SUCCESS", "_SUCCESS" in _o6))
_c6 = hadoop(f"hdfs dfs -cat {HDFS_OUT}/wordcount/part-*", hien_thi=False)
_d6 = {p[0]: int(p[1]) for p in (d.split("\t") for d in _c6.strip().split("\n")) if len(p) == 2}
_muc.append(("D6  Kết quả YARN khớp Python (373 từ)", len(_d6) == 373 and _d6.get("một") == 43))

# --- D7
_o7 = hadoop(f"hdfs dfs -ls {HDFS_OUT}/revenue", hien_thi=False)
_muc.append(("D7  Job doanh thu có tệp _SUCCESS", "_SUCCESS" in _o7))
_c7 = hadoop(f"hdfs dfs -cat {HDFS_OUT}/revenue/part-*", hien_thi=False)
_d7 = {p[0]: (int(p[1]), int(p[2])) for p in (d.split("\t") for d in _c7.strip().split("\n")) if len(p) == 3}
_muc.append(("D7  Doanh thu 6 ngành hàng khớp đáp án", _d7 == CHUAN_REVENUE))

# --- D8
_muc.append(("D8  Đã ghi outputs/lab4_timings.json", (OUTPUT_DIR / "lab4_timings.json").exists()))

_dat = 0
for _ten, _ok in _muc:
    print(f"  {'[DAT] ' if _ok else '[    ]'}  {_ten}")
    _dat += bool(_ok)

print("-" * 74)
print(f"HOÀN THÀNH: {_dat}/{len(_muc)} mục  ({_dat/len(_muc)*100:.0f}%)")
print("=" * 74)
if _dat == len(_muc):
    print("Chúc mừng! Bạn đã hoàn thành trọn vẹn Buổi 04.")
    print("Bạn đã lưu dữ liệu trên hệ thống tệp phân tán, điều phối tài nguyên bằng YARN,")
    print("và tính toán bằng MapReduce — ba nền tảng của toàn bộ hệ sinh thái Big Data.")
    print("\nĐừng quên nộp outputs/lab4_timings.json — Buổi 05 sẽ đọc nó để so sánh với Spark.")
else:
    print("Còn mục chưa xong (ô trống [    ]). Hoàn thiện rồi chạy lại ô này.")
print("=" * 74)
